# 文本分类实例

## Step1 导入相关包

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step2 加载数据

In [8]:
import pandas as pd

data = pd.read_csv("./ChnSentiCorp_htl_all.csv")
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


In [9]:
data = data.dropna()
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


## Step3 创建Dataset

In [5]:
from torch.utils.data import Dataset

class MyDataset(Dataset):

    def __init__(self) -> None:
        super().__init__()
        self.data = pd.read_csv("./ChnSentiCorp_htl_all.csv")
        self.data = self.data.dropna()

    def __getitem__(self, index):
        return self.data.iloc[index]["review"], self.data.iloc[index]["label"]
    
    def __len__(self):
        return len(self.data)

In [10]:
dataset = MyDataset()
for i in range(5):
    print(dataset[i])

('距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较为简单.', 1)
('商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!', 1)
('早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。', 1)
('宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小，但加上低价位因素，还是无超所值的；环境不错，就在小胡同内，安静整洁，暖气好足-_-||。。。呵还有一大优势就是从宾馆出发，步行不到十分钟就可以到梅兰芳故居等等，京味小胡同，北海距离好近呢。总之，不错。推荐给节约消费的自助游朋友~比较划算，附近特色小吃很多~', 1)
('CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风', 1)


## Step4 划分数据集

In [11]:
from torch.utils.data import random_split


trainset, validset = random_split(dataset, lengths=[0.9, 0.1])
len(trainset), len(validset)

(6989, 776)

In [12]:
for i in range(10):
    print(trainset[i])

('感觉还是不错的，房间很大，也很干净，衣柜足够用。早餐还可以，只是感觉应提高点儿质量。', 1)
('住的是大床房，和先前筒子们介绍的情况差不多，会免费升级一下房间，有两瓶免费的可乐送。对该酒店服务总体评价也就中规中矩四个字吧，不过，的确评到五星是有点过头了。四星差不多了。当然，性价比而言，还算可以，毕竟花的不是标准五星的价。对该店算是有意见的话，那就是实在老了点，虽然看得出花过点功夫重新装修，但偶还是不够喜欢。另外，早餐也算基本满意吧，品种数量还可以，味道soso啦。软件方面还可以，要地图有免费地图送上门，草鞋托也算有点小意思，基本功还算到位。', 1)
('如家的定位还是很不错的，我出差还是基本订如家。这次在十一经路店遇到了麻烦，在一楼餐厅吃饭时，钱夹被偷了！本来出门很小心的，看到餐厅里只有很少几位住店客人就餐，也就放松了警惕，把外套放在了椅子上，结果被贼盯上了，做到我后面掏了我的口袋（后悔呀，座到最后一排，恐怕贼就不好下手了！）。小服务员（估计不到20）觉出什么来了，好心提醒我要给我套衣服，我还觉得奇怪，平常没套过呀（活该破财，人家暗示我，我都没反应）。大家再来天津这个店，要小心了！', 1)
('酒店地理位置距离南陀岛海洋浴场很近，服务质量优良，大堂的自助晚餐超值优惠，值得一去。房间设施舒适，宽敞。', 1)
('开始订了富凯，虽然住在8楼双号，但是早上7点的时候就能听到下面的国歌声，早操音乐，实在太吵，第二晚就换到凯莱。我觉得房间很不错。床也很舒服。住在23楼很安静。除了不知道是不是新装修过的原因，走廊里有种难闻的气味，其他都好！而且12.31号当天入住的价格318比富凯还便宜20，房间要好上很多倍！（不过刚才看最便宜的房价已经398了，难道涨价了？）', 1)
('如果作为四星级的话，可以打70分吧。交通还是挺便利的。早餐也不错：）宾馆反馈2008年7月9日：欣喜地知会：2007年9月，广州天伦万怡大酒店顺利通过国家旅游局饭店星级评定委员会星级评定，荣膺五星级饭店。感谢阁下对我店早餐的赞赏，我们会保持好的出品，不断创新。', 1)
('整体环境比较不错,离景点也很近,很方便.适合家庭旅游居住', 1)
('宾馆竟然不能提供网络调试服务，我做了一次兼职的网管，呵呵！很有面子，去得那天天气冷，刚开始有点不太适应，要空调遥控器，说现在不提供；要加被子，说没有

## Step5 创建Dataloader

In [13]:
import torch

tokenizer = AutoTokenizer.from_pretrained("rbt3")

def collate_func(batch):
    texts, labels = [], []
    for item in batch:
        texts.append(item[0])
        labels.append(item[1])
    inputs = tokenizer(texts, max_length=128, padding="max_length", truncation=True, return_tensors="pt")
    inputs["labels"] = torch.tensor(labels)
    return inputs

In [ ]:
from torch.utils.data import DataLoader

trainloader = DataLoader(trainset, batch_size=32, shuffle=True, collate_fn=collate_func)
validloader = DataLoader(validset, batch_size=64, shuffle=False, collate_fn=collate_func)

In [15]:
next(enumerate(validloader))[1]

{'input_ids': tensor([[ 101, 2791, 7313,  ...,    0,    0,    0],
        [ 101, 2769,  812,  ...,    0,    0,    0],
        [ 101, 2769, 3221,  ..., 4717, 4708,  102],
        ...,
        [ 101, 6983, 2421,  ...,    0,    0,    0],
        [ 101, 2345, 1168,  ...,    0,    0,    0],
        [ 101, 1071, 2124,  ...,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0,
        1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0,
        1, 0, 1

## Step6 创建模型及优化器

In [16]:
from torch.optim import Adam

model = AutoModelForSequenceClassification.from_pretrained("rbt3")

if torch.cuda.is_available():
    model = model.cuda()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at rbt3 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
optimizer = Adam(model.parameters(), lr=2e-5)

## Step7 训练与验证

In [18]:
def evaluate():
    model.eval()
    acc_num = 0
    with torch.inference_mode():
        for batch in validloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            output = model(**batch)
            pred = torch.argmax(output.logits, dim=-1)
            acc_num += (pred.long() == batch["labels"].long()).float().sum()
    return acc_num / len(validset)

def train(epoch=3, log_step=100):
    global_step = 0
    for ep in range(epoch):
        model.train()
        for batch in trainloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            optimizer.zero_grad()
            output = model(**batch)
            output.loss.backward()
            optimizer.step()
            if global_step % log_step == 0:
                print(f"ep: {ep}, global_step: {global_step}, loss: {output.loss.item()}")
            global_step += 1
        acc = evaluate()
        print(f"ep: {ep}, acc: {acc}")

## Step8 模型训练

In [19]:
train()

ep: 0, global_step: 0, loss: 0.6879236698150635
ep: 0, global_step: 100, loss: 0.4248634874820709
ep: 0, global_step: 200, loss: 0.25124743580818176
ep: 0, acc: 0.8672680258750916
ep: 1, global_step: 300, loss: 0.2913534343242645
ep: 1, global_step: 400, loss: 0.13272695243358612
ep: 1, acc: 0.8891752362251282
ep: 2, global_step: 500, loss: 0.21128679811954498
ep: 2, global_step: 600, loss: 0.0991872251033783
ep: 2, acc: 0.9007731676101685


## Step9 模型预测

In [25]:
sen = "我觉得这家酒店不错，饭很好吃！"
id2_label = {0: "差评！", 1: "好评！"}
model.eval()
with torch.inference_mode():
    inputs = tokenizer(sen, return_tensors="pt")
    inputs = {k: v.cuda() for k, v in inputs.items()}
    logits = model(**inputs).logits
    pred = torch.argmax(logits, dim=-1)
    print(f"输入：{sen}\n模型预测结果:{id2_label.get(pred.item())}")

输入：我觉得这家酒店不错，饭很好吃！
模型预测结果:好评！


In [26]:
from transformers import pipeline

model.config.id2label = id2_label
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

In [27]:
pipe(sen)

[{'label': '好评！', 'score': 0.9899283051490784}]